In [4]:
# 04 XGBoost Training

"""01_eda.ipynbThis notebook performs:
- dataset loading from `data/features.parquet`
- user-level train/test split
- Optuna hyperparameter tuning
- XGBoost training
- Macro F1 + AUC-ROC evaluation
- export of `models/xgb_full.joblib`
- export of `reports/xgb_evaluation.md`"""

'01_eda.ipynbThis notebook performs:\n- dataset loading from `data/features.parquet`\n- user-level train/test split\n- Optuna hyperparameter tuning\n- XGBoost training\n- Macro F1 + AUC-ROC evaluation\n- export of `models/xgb_full.joblib`\n- export of `reports/xgb_evaluation.md`'

In [11]:
!pip install optuna xgboost shap scikit-learn pandas joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.8/556.8 kB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 33.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 33.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 19.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [optuna]10/11 [optuna]]my]


In [12]:
from pathlib import Path
import json

import joblib
import optuna
import pandas as pd
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier

In [13]:
ROOT = Path.cwd().resolve().parents[1]  # because you're inside notebooks/

DATA_PATH = ROOT / "data" / "features.parquet"
MODEL_PATH = ROOT / "models" / "xgb_full.joblib"
REPORT_PATH = ROOT / "reports" / "xgb_evaluation.md"

print("ROOT:", ROOT)
print("DATA:", DATA_PATH)

LABEL_MAP = {"LOW": 0, "MODERATE": 1, "HIGH": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

FEATURE_COLUMNS = [
    "n_interactions",
    "accuracy",
    "accuracy_pct",
    "incorrect_rate",
    "avg_elapsed_time",
    "median_elapsed_time",
    "std_elapsed_time",
    "p90_elapsed_time",
    "long_response_rate",
    "retry_rate",
    "wrong_streak_max",
    "efficiency_score",
    "error_burden",
    "retry_burden",
    "time_pressure_index",
    "struggle_index",
    "consistency_index",
    "pace_ratio",
    "elapsed_range_proxy",
]

print("Data path:", DATA_PATH)
print("Model path:", MODEL_PATH)

ROOT: /Users/jigyasa/Vscode-me/cognio
DATA: /Users/jigyasa/Vscode-me/cognio/data/features.parquet
Data path: /Users/jigyasa/Vscode-me/cognio/data/features.parquet
Model path: /Users/jigyasa/Vscode-me/cognio/models/xgb_full.joblib


In [15]:
ROOT = Path.cwd().resolve().parents[1]   # goes to /cognio
DATA_PATH = ROOT / "ml" / "data" / "features.parquet"
MODEL_PATH = ROOT / "ml" / "models" / "xgb_full.joblib"
REPORT_PATH = ROOT / "ml" / "reports" / "xgb_evaluation.md"

print("DATA:", DATA_PATH)

DATA: /Users/jigyasa/Vscode-me/cognio/ml/data/features.parquet


In [16]:
print(DATA_PATH.exists())

True


In [17]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing file: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH).copy()
print("Shape:", df.shape)
print(df.head())

Shape: (188, 25)
  user_id  session_num session_id  n_interactions  accuracy  accuracy_pct  \
0       1            0       1_s0               3       0.0           0.0   
1       2            0       2_s0               3       0.0           0.0   
2       3            0       3_s0               3       0.0           0.0   
3       4            0       4_s0               3       0.0           0.0   
4       5            0       5_s0               3       0.0           0.0   

   incorrect_rate  avg_elapsed_time  median_elapsed_time  std_elapsed_time  \
0             1.0         26.666667                 25.0          8.653837   
1             1.0         35.000000                 27.0         13.490738   
2             1.0         45.666667                 36.0         15.839472   
3             1.0         31.000000                 29.0          8.286535   
4             1.0         33.333333                 22.0         18.190352   

   ...  efficiency_score  error_burden  retry_burde

In [18]:
required_cols = FEATURE_COLUMNS + ["user_id", "strain_level", "target"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=required_cols).copy()

print("Training rows:", len(df))
print("Unique users:", df["user_id"].nunique())
print("\nClass distribution:")
print(df["strain_level"].value_counts())

Training rows: 188
Unique users: 187

Class distribution:
strain_level
HIGH        63
LOW         63
MODERATE    62
Name: count, dtype: int64


In [19]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = df["user_id"]

train_idx, test_idx = next(splitter.split(df, groups=groups))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train users:", train_df["user_id"].nunique())
print("Test users:", test_df["user_id"].nunique())

print("\nTrain classes:")
print(train_df["strain_level"].value_counts())

print("\nTest classes:")
print(test_df["strain_level"].value_counts())

Train rows: 150
Test rows: 38
Train users: 149
Test users: 38

Train classes:
strain_level
HIGH        51
MODERATE    50
LOW         49
Name: count, dtype: int64

Test classes:
strain_level
LOW         14
HIGH        12
MODERATE    12
Name: count, dtype: int64


In [20]:
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df["target"].astype(int)

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df["target"].astype(int)

print(X_train.shape, X_test.shape)

(150, 19) (38, 19)


In [21]:
def build_model(trial: optuna.Trial) -> XGBClassifier:
    return XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 80, 300),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 8),
        gamma=trial.suggest_float("gamma", 0.0, 3.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.0, 3.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.5, 5.0),
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        tree_method="hist",
    )


def objective(trial: optuna.Trial) -> float:
    model = build_model(trial)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=False)

print("Best trial score:", study.best_value)
print("Best params:")
print(json.dumps(study.best_params, indent=2))

[I 2026-04-22 17:03:04,917] A new study created in memory with name: no-name-b0d06d11-7c6a-4c01-a5c7-661ac44f629c
[I 2026-04-22 17:03:05,106] Trial 0 finished with value: 0.9475205417234402 and parameters: {'n_estimators': 148, 'max_depth': 7, 'learning_rate': 0.08138548782952389, 'subsample': 0.9829448859107618, 'colsample_bytree': 0.829552673481714, 'min_child_weight': 7, 'gamma': 0.04351153838259225, 'reg_alpha': 1.9491693341284133, 'reg_lambda': 2.007055006281302}. Best is trial 0 with value: 0.9475205417234402.
[I 2026-04-22 17:03:05,456] Trial 1 finished with value: 0.9475205417234402 and parameters: {'n_estimators': 199, 'max_depth': 8, 'learning_rate': 0.01255664057438344, 'subsample': 0.8324714860891697, 'colsample_bytree': 0.9224065307395991, 'min_child_weight': 1, 'gamma': 0.21823772565515442, 'reg_alpha': 0.3449381811172778, 'reg_lambda': 4.44431845163216}. Best is trial 0 with value: 0.9475205417234402.
[I 2026-04-22 17:03:05,584] Trial 2 finished with value: 0.94752054172

Best trial score: 0.9475205417234402
Best params:
{
  "n_estimators": 148,
  "max_depth": 7,
  "learning_rate": 0.08138548782952389,
  "subsample": 0.9829448859107618,
  "colsample_bytree": 0.829552673481714,
  "min_child_weight": 7,
  "gamma": 0.04351153838259225,
  "reg_alpha": 1.9491693341284133,
  "reg_lambda": 2.007055006281302
}


In [22]:
best_model = build_model(study.best_trial)
best_model.fit(X_train, y_train)

preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)

macro_f1 = f1_score(y_test, preds, average="macro")
auc_roc = roc_auc_score(y_test, probs, multi_class="ovr")

report = classification_report(
    y_test,
    preds,
    target_names=["LOW", "MODERATE", "HIGH"],
    digits=4,
    zero_division=0,
)

print("Macro F1:", round(float(macro_f1), 4))
print("AUC-ROC (OVR):", round(float(auc_roc), 4))
print("\nClassification Report:\n")
print(report)

Macro F1: 0.9475
AUC-ROC (OVR): 0.9927

Classification Report:

              precision    recall  f1-score   support

         LOW     1.0000    0.9286    0.9630        14
    MODERATE     0.8571    1.0000    0.9231        12
        HIGH     1.0000    0.9167    0.9565        12

    accuracy                         0.9474        38
   macro avg     0.9524    0.9484    0.9475        38
weighted avg     0.9549    0.9474    0.9483        38



In [23]:
artifact = {
    "model": best_model,
    "feature_columns": FEATURE_COLUMNS,
    "label_map": LABEL_MAP,
    "inverse_label_map": INV_LABEL_MAP,
    "best_params": study.best_params,
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, MODEL_PATH)

print("Saved model to:", MODEL_PATH)

Saved model to: /Users/jigyasa/Vscode-me/cognio/ml/models/xgb_full.joblib


In [24]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

REPORT_PATH.write_text(
    "# XGBoost Evaluation Report\n\n"
    f"Training rows used: {len(df)}\n\n"
    f"Unique users: {df['user_id'].nunique()}\n\n"
    f"Train split rows: {len(train_df)}\n\n"
    f"Test split rows: {len(test_df)}\n\n"
    f"Train users: {train_df['user_id'].nunique()}\n\n"
    f"Test users: {test_df['user_id'].nunique()}\n\n"
    f"Macro F1: {macro_f1:.4f}\n\n"
    f"AUC-ROC (OVR): {auc_roc:.4f}\n\n"
    "## Best Optuna Parameters\n\n"
    f"```json\n{json.dumps(study.best_params, indent=2)}\n```\n\n"
    "## Classification Report\n\n"
    f"```\n{report}\n```\n",
    encoding="utf-8",
)

print("Saved report to:", REPORT_PATH)

Saved report to: /Users/jigyasa/Vscode-me/cognio/ml/reports/xgb_evaluation.md


In [26]:
LOG_PATH = ROOT / "ml" / "reports" / "retrain_log.json"

In [27]:
LOG_PATH.write_text(
    json.dumps(
        {
            "status": "success",
            "model_path": str(MODEL_PATH),
            "rows_used": int(len(df)),
            "unique_users": int(df["user_id"].nunique()),
            "train_rows": int(len(train_df)),
            "test_rows": int(len(test_df)),
            "train_users": int(train_df["user_id"].nunique()),
            "test_users": int(test_df["user_id"].nunique()),
            "macro_f1": round(float(macro_f1), 4),
            "auc_roc_ovr": round(float(auc_roc), 4),
            "feature_columns": FEATURE_COLUMNS,
            "class_distribution": df["strain_level"].value_counts().to_dict(),
            "best_params": study.best_params,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved retrain log to:", LOG_PATH)

Saved retrain log to: /Users/jigyasa/Vscode-me/cognio/ml/reports/retrain_log.json


In [28]:
sample_row = X_test.iloc[[0]].copy()
sample_probs = best_model.predict_proba(sample_row)[0]
sample_pred = int(best_model.predict(sample_row)[0])

print("Predicted class:", INV_LABEL_MAP[sample_pred])
print("Probabilities:", sample_probs)
print("Top 3 highest probability classes ranked:")
ranked = sorted(
    [(INV_LABEL_MAP[i], float(p)) for i, p in enumerate(sample_probs)],
    key=lambda x: x[1],
    reverse=True
)
print(ranked)

Predicted class: HIGH
Probabilities: [0.08399997 0.13685611 0.7791439 ]
Top 3 highest probability classes ranked:
[('HIGH', 0.7791439294815063), ('MODERATE', 0.1368561089038849), ('LOW', 0.08399996906518936)]
